# Active Stations, Electric Bike, Membership

In [1]:
import networkx as nx
import osmnx as ox
import numpy as np
import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import ScrollZoomToggler
import matplotlib.pyplot as plt
import shapely
from glob import glob
from datetime import timedelta
import osmnx.settings as settings
import osmnx.features as features

ox.__version__

settings.cache_folder = "/tmp/cache"


In [37]:
from tqdm import tqdm
import pandas as pd
import glob

# Define the function for borough assignment based on lat/lng
def get_borough(lat, lng):
    if 40.70 <= lat <= 40.88 and -74.02 <= lng <= -73.90:
        return 'Manhattan'
    elif 40.57 <= lat <= 40.73 and -74.04 <= lng <= -73.85:
        return 'Brooklyn'
    elif 40.54 <= lat <= 40.80 and -73.95 <= lng <= -73.70:
        return 'Queens'
    elif 40.79 <= lat <= 40.91 and -73.93 <= lng <= -73.80:
        return 'Bronx'
    elif 40.49 <= lat <= 40.65 and -74.25 <= lng <= -74.05:
        return 'Staten Island'
    else:
        return 'Other'

# Define the path to your files
path = '/Users/amyxqc/Desktop/amy_bikenycfall2024/Data/CitiBike/*.csv'

# Initialize an empty DataFrame to store the aggregated results
final_activestations_data = pd.DataFrame()

# Loop through each CSV file in the specified folder
for file in tqdm(sorted(glob.glob(path)), desc="Processing Files"):
    print(f"Processing file: {file}")
    
    # Load the data
    df = pd.read_csv(file)
    
    # Ensure the required columns are present
    required_columns = ['started_at', 'start_lat', 'start_lng', 'start_station_id', 'end_station_id']
    if not all(col in df.columns for col in required_columns):
        print(f"Missing columns in file {file}. Skipping...")
        continue

    # Convert 'started_at' to datetime
    df['started_at'] = pd.to_datetime(df['started_at'], errors='coerce')
    df = df.dropna(subset=['started_at'])  # Drop rows with invalid dates
    
    # Filter for data starting from 2021
    df = df[df['started_at'] >= '2021-01-01']
    
    # Apply the get_borough function to each row to determine the borough
    df['borough'] = df.apply(lambda row: get_borough(row['start_lat'], row['start_lng']), axis=1)
    
    # Extract year and week number
    df['year'] = df['started_at'].dt.year
    df['week_number'] = df['started_at'].dt.isocalendar().week
    
    # Fix for week 53: Assign to week 1 if in January of the new year
    df.loc[(df['week_number'] == 53) & (df['started_at'].dt.month == 1), 'week_number'] = 1

    # Aggregate the data to weekly summaries
    weekly_aggregated = df.groupby(['year', 'week_number', 'borough']).apply(
        lambda x: pd.unique(pd.concat([x['start_station_id'], x['end_station_id']])).size
    ).reset_index(name='active_stations')

    # Append the aggregated data directly to the final DataFrame
    final_activestations_data = pd.concat([final_activestations_data, weekly_aggregated], ignore_index=True)

# Ensure the final DataFrame starts at week 1
final_activestations_data = final_activestations_data[final_activestations_data['week_number'] >= 1]

# Display the final combined DataFrame
print(final_activestations_data.head())


Processing Files:   0%|          | 0/143 [00:00<?, ?it/s]

Processing file: /Users/amyxqc/Desktop/amy_bikenycfall2024/Data/CitiBike/202101-citibike-tripdata_1.csv


/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_69947/1101183331.py:31: DtypeWarning: Columns (5,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)
/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_69947/1101183331.py:40: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['started_at'] = pd.to_datetime(df['started_at'], errors='coerce')
/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_69947/1101183331.py:57: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  weekly_aggregated = df.group

Processing file: /Users/amyxqc/Desktop/amy_bikenycfall2024/Data/CitiBike/202101-citibike-tripdata_2.csv


/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_69947/1101183331.py:57: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  weekly_aggregated = df.groupby(['year', 'week_number', 'borough']).apply(
Processing Files:   1%|▏         | 2/143 [00:07<07:37,  3.24s/it]

...

Processing file: /Users/amyxqc/Desktop/amy_bikenycfall2024/Data/CitiBike/202410-citibike-tripdata.csv


/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_69947/1101183331.py:57: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  weekly_aggregated = df.groupby(['year', 'week_number', 'borough']).apply(
Processing Files:  99%|█████████▉| 142/143 [15:46<00:05,  5.51s/it]

Processing file: /Users/amyxqc/Desktop/amy_bikenycfall2024/Data/CitiBike/202411-citibike-tripdata.csv


/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_69947/1101183331.py:57: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  weekly_aggregated = df.groupby(['year', 'week_number', 'borough']).apply(
Processing Files: 100%|██████████| 143/143 [15:46<00:00,  6.62s/it]

   year  week_number    borough  active_stations
0  2021            1      Bronx              178
1  2021            1   Brooklyn             1115
2  2021            1  Manhattan             2110
3  2021            2      Bronx              166
4  2021            2   Brooklyn             1119


In [67]:
print(final_activestations_data)

      year    borough  active_stations
0     2021      Bronx              178
1     2021   Brooklyn             1115
2     2021  Manhattan             2110
12    2021      Bronx               76
13    2021   Brooklyn              500
...    ...        ...              ...
3621  2024      Other              114
3622  2024   Brooklyn               93
3623  2024      Other              111
3624  2024   Brooklyn               98
3625  2024      Other               99

[3626 rows x 3 columns]


In [71]:
from tqdm import tqdm
import pandas as pd
import glob

# Define the function for borough assignment based on lat/lng
def get_borough(lat, lng):
    if 40.70 <= lat <= 40.88 and -74.02 <= lng <= -73.90:
        return 'Manhattan'
    elif 40.57 <= lat <= 40.73 and -74.04 <= lng <= -73.85:
        return 'Brooklyn'
    elif 40.54 <= lat <= 40.80 and -73.95 <= lng <= -73.70:
        return 'Queens'
    elif 40.79 <= lat <= 40.91 and -73.93 <= lng <= -73.80:
        return 'Bronx'
    elif 40.49 <= lat <= 40.65 and -74.25 <= lng <= -74.05:
        return 'Staten Island'
    else:
        return 'Other'

# Define the path to your files
path = '/Users/amyxqc/Desktop/amy_bikenycfall2024/Data/CitiBike/*.csv'

# Initialize an empty DataFrame to store the aggregated results
final_activestations_data = pd.DataFrame()

# Loop through each CSV file in the specified folder
for file in tqdm(sorted(glob.glob(path)), desc="Processing Files"):
    print(f"Processing file: {file}")
    
    # Load the data
    df = pd.read_csv(file)
    
    # Ensure the required columns are present
    required_columns = ['started_at', 'start_lat', 'start_lng', 'start_station_id', 'end_station_id']
    if not all(col in df.columns for col in required_columns):
        print(f"Missing columns in file {file}. Skipping...")
        continue

    # Convert 'started_at' to datetime
    df['started_at'] = pd.to_datetime(df['started_at'], errors='coerce')
    df = df.dropna(subset=['started_at'])  # Drop rows with invalid dates
    
    # Filter for data starting from 2021
    df = df[df['started_at'] >= '2021-01-01']
    
    # Apply the get_borough function to each row to determine the borough
    df['borough'] = df.apply(lambda row: get_borough(row['start_lat'], row['start_lng']), axis=1)
    
    # Extract year and week number
    df['year'] = df['started_at'].dt.year
    df['week_number'] = df['started_at'].dt.isocalendar().week
    
    # Fix for week 53: Assign to week 1 if in January of the new year
    df.loc[(df['week_number'] == 53) & (df['started_at'].dt.month == 1), 'week_number'] = 1
    
    # Calculate cumulative week number
    df['cumulative_week_number'] = (
        (df['year'] - df['year'].min()) * 52
    ) + df['week_number']
    
    # Aggregate the data to weekly summaries
    weekly_aggregated = df.groupby(['year', 'cumulative_week_number', 'borough']).apply(
        lambda x: pd.unique(pd.concat([x['start_station_id'], x['end_station_id']])).size
    ).reset_index(name='active_stations')

    # Append the aggregated data directly to the final DataFrame
    final_activestations_data = pd.concat([final_activestations_data, weekly_aggregated], ignore_index=True)

# Drop duplicates and reset index
final_activestations_data.drop_duplicates(inplace=True)
final_activestations_data.reset_index(drop=True, inplace=True)

# Display the final combined DataFrame
print(final_activestations_data)


Processing Files:   0%|          | 0/143 [00:00<?, ?it/s]

Processing file: /Users/amyxqc/Desktop/amy_bikenycfall2024/Data/CitiBike/202101-citibike-tripdata_1.csv


/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_69947/3006853845.py:31: DtypeWarning: Columns (5,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)
/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_69947/3006853845.py:40: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['started_at'] = pd.to_datetime(df['started_at'], errors='coerce')
/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_69947/3006853845.py:62: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  weekly_aggregated = df.group

Processing file: /Users/amyxqc/Desktop/amy_bikenycfall2024/Data/CitiBike/202101-citibike-tripdata_2.csv


/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_69947/3006853845.py:62: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  weekly_aggregated = df.groupby(['year', 'cumulative_week_number', 'borough']).apply(
Processing Files:   1%|▏         | 2/143 [00:07<07:47,  3.32s/it]

...

Processing file: /Users/amyxqc/Desktop/amy_bikenycfall2024/Data/CitiBike/202410-citibike-tripdata.csv


/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_69947/3006853845.py:62: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  weekly_aggregated = df.groupby(['year', 'cumulative_week_number', 'borough']).apply(
Processing Files:  99%|█████████▉| 142/143 [15:51<00:05,  5.46s/it]

Processing file: /Users/amyxqc/Desktop/amy_bikenycfall2024/Data/CitiBike/202411-citibike-tripdata.csv


/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_69947/3006853845.py:62: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  weekly_aggregated = df.groupby(['year', 'cumulative_week_number', 'borough']).apply(
Processing Files: 100%|██████████| 143/143 [15:51<00:00,  6.66s/it]

      year  cumulative_week_number    borough  active_stations
0     2021                       1      Bronx              178
1     2021                       1   Brooklyn             1115
2     2021                       1  Manhattan             2110
3     2021                       2      Bronx              166
4     2021                       2   Brooklyn             1119
...    ...                     ...        ...              ...
3323  2024                      46      Other              114
3324  2024                      47   Brooklyn               93
3325  2024                      47      Other              111
3326  2024                      48   Brooklyn               98
3327  2024                      48      Other               99

[3328 rows x 4 columns]


In [86]:
from tqdm import tqdm
import pandas as pd
import glob

def get_borough(lat, lng):
    if 40.70 <= lat <= 40.88 and -74.02 <= lng <= -73.90:
        return 'Manhattan'
    elif 40.57 <= lat <= 40.73 and -74.04 <= lng <= -73.85:
        return 'Brooklyn'
    elif 40.54 <= lat <= 40.80 and -73.95 <= lng <= -73.70:
        return 'Queens'
    elif 40.79 <= lat <= 40.91 and -73.93 <= lng <= -73.80:
        return 'Bronx'
    elif 40.49 <= lat <= 40.65 and -74.25 <= lng <= -74.05:
        return 'Staten Island'
    else:
        return 'Other'

path = '/Users/amyxqc/Desktop/amy_bikenycfall2024/Data/CitiBike/*.csv'
final_activestations_data = pd.DataFrame()

for file in tqdm(sorted(glob.glob(path)), desc="Processing Files"):
    print(f"Processing file: {file}")
    df = pd.read_csv(file)
    df = df.drop_duplicates()
    required_columns = ['started_at', 'start_lat', 'start_lng', 'start_station_id', 'end_station_id']
    if not all(col in df.columns for col in required_columns):
        print(f"Missing columns in file {file}. Skipping...")
        continue

    df['started_at'] = pd.to_datetime(df['started_at'], errors='coerce')
    df = df.dropna(subset=['started_at'])
    df = df[df['started_at'] >= '2021-01-01']
    df['borough'] = df.apply(lambda row: get_borough(row['start_lat'], row['start_lng']), axis=1)
    df = df.dropna(subset=['start_station_id', 'end_station_id'])

    df['year'] = df['started_at'].dt.year
    df['week_number'] = df['started_at'].dt.isocalendar().week
    df['cumulative_week_number'] = (
        (df['year'] - df['year'].min()) * 52
    ) + df['week_number']

    weekly_aggregated = df.groupby(['year', 'cumulative_week_number', 'borough']).apply(
        lambda x: pd.Series(pd.concat([x['start_station_id'], x['end_station_id']]).unique()).size
    ).reset_index(name='active_stations')

    final_activestations_data = pd.concat([final_activestations_data, weekly_aggregated], ignore_index=True)

final_activestations_data.drop_duplicates(inplace=True)
final_activestations_data.reset_index(drop=True, inplace=True)
print(final_activestations_data)

Processing Files:   0%|          | 0/143 [00:00<?, ?it/s]

Processing file: /Users/amyxqc/Desktop/amy_bikenycfall2024/Data/CitiBike/202101-citibike-tripdata_1.csv


/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_69947/4054391358.py:24: DtypeWarning: Columns (5,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)
/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_69947/4054391358.py:31: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['started_at'] = pd.to_datetime(df['started_at'], errors='coerce')
/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_69947/4054391358.py:43: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  weekly_aggregated = df.group

Processing file: /Users/amyxqc/Desktop/amy_bikenycfall2024/Data/CitiBike/202101-citibike-tripdata_2.csv


/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_69947/4054391358.py:43: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  weekly_aggregated = df.groupby(['year', 'cumulative_week_number', 'borough']).apply(
Processing Files:   1%|▏         | 2/143 [00:08<08:36,  3.66s/it]

...

Processing file: /Users/amyxqc/Desktop/amy_bikenycfall2024/Data/CitiBike/202106-citibike-tripdata_4.csv


/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_69947/4054391358.py:24: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)
/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_69947/4054391358.py:43: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  weekly_aggregated = df.groupby(['year', 'cumulative_week_number', 'borough']).apply(
Processing Files:  10%|█         | 15/143 [01:32<12:13,  5.73s/it]

Processing file: /Users/amyxqc/Desktop/amy_bikenycfall2024/Data/CitiBike/202107-citibike-tripdata_1.csv


/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_69947/4054391358.py:24: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)
Processing Files:  10%|█         | 15/143 [01:40<14:21,  6.73s/it]


KeyboardInterrupt: 

In [97]:
import pandas as pd
import glob
from tqdm import tqdm

# Define the function for borough assignment based on lat/lng
def get_borough(lat, lng):
    if 40.70 <= lat <= 40.88 and -74.02 <= lng <= -73.90:
        return 'Manhattan'
    elif 40.57 <= lat <= 40.73 and -74.04 <= lng <= -73.85:
        return 'Brooklyn'
    elif 40.54 <= lat <= 40.80 and -73.95 <= lng <= -73.70:
        return 'Queens'
    elif 40.79 <= lat <= 40.91 and -73.93 <= lng <= -73.80:
        return 'Bronx'
    elif 40.49 <= lat <= 40.65 and -74.25 <= lng <= -74.05:
        return 'Staten Island'
    else:
        return 'Other'

# Define the path to your files
path = '/Users/amyxqc/Desktop/amy_bikenycfall2024/Data/CitiBike/*.csv'

# Initialize an empty DataFrame for the results
unique_stations_by_borough = pd.DataFrame()

# Process each file
for file in tqdm(sorted(glob.glob(path)), desc="Processing Files"):
    print(f"Processing file: {file}")
    df = pd.read_csv(file)
    df = df.drop_duplicates()

    required_columns = ['start_lat', 'start_lng', 'start_station_id', 'end_station_id']
    if not all(col in df.columns for col in required_columns):
        print(f"Missing columns in file {file}. Skipping...")
        continue

    # Assign borough to start stations
    df['start_borough'] = df.apply(lambda row: get_borough(row['start_lat'], row['start_lng']), axis=1)

    # Assign borough to end stations
    df['end_borough'] = df.apply(lambda row: get_borough(row['end_lat'], row['end_lng']), axis=1)

    # Combine both start and end stations into one DataFrame
    start_stations = df[['start_station_id', 'start_borough']].rename(columns={'start_station_id': 'station_id', 'start_borough': 'borough'})
    end_stations = df[['end_station_id', 'end_borough']].rename(columns={'end_station_id': 'station_id', 'end_borough': 'borough'})

    # Concatenate start and end stations and drop duplicates
    all_stations = pd.concat([start_stations, end_stations]).drop_duplicates()

    # Count unique station IDs per borough
    borough_station_counts = all_stations.groupby('borough')['station_id'].nunique().reset_index(name='unique_stations')

    # Append results to the final DataFrame
    unique_stations_by_borough = pd.concat([unique_stations_by_borough, borough_station_counts], ignore_index=True)

# Aggregate results across all files
final_unique_stations_by_borough = unique_stations_by_borough.groupby('borough')['unique_stations'].sum().reset_index()

# Display the results
print(final_unique_stations_by_borough)


Processing Files:   0%|          | 0/143 [00:00<?, ?it/s]

Processing file: /Users/amyxqc/Desktop/amy_bikenycfall2024/Data/CitiBike/202101-citibike-tripdata_1.csv


/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_69947/966383055.py:29: DtypeWarning: Columns (5,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)
Processing Files:   1%|          | 1/143 [00:11<26:06, 11.03s/it]

Processing file: /Users/amyxqc/Desktop/amy_bikenycfall2024/Data/CitiBike/202101-citibike-tripdata_2.csv


Processing Files:   1%|▏         | 2/143 [00:12<12:18,  5.24s/it]

...

Processing file: /Users/amyxqc/Desktop/amy_bikenycfall2024/Data/CitiBike/202410-citibike-tripdata.csv


Processing Files:  99%|█████████▉| 142/143 [24:27<00:08,  8.56s/it]

Processing file: /Users/amyxqc/Desktop/amy_bikenycfall2024/Data/CitiBike/202411-citibike-tripdata.csv


Processing Files: 100%|██████████| 143/143 [24:28<00:00, 10.27s/it]

         borough  unique_stations
0          Bronx            24171
1       Brooklyn           125977
2      Manhattan           297454
3          Other             1715
4         Queens            15479
5  Staten Island               10


In [104]:
import pandas as pd
import glob
from tqdm import tqdm

# Define the path to your files
path = '/Users/amyxqc/Desktop/amy_bikenycfall2024/Data/CitiBike/*.csv'

# Initialize a dictionary to store unique station IDs for each year
unique_stations_per_year = {}

# Loop through each year from 2021 onward
for year in range(2021, 2025):  # Adjust the range based on your dataset
    print(f"Processing year: {year}")
    
    # Initialize an empty DataFrame for the current year
    yearly_data = pd.DataFrame()
    
    # Loop through each CSV file in the directory
    for file in tqdm(sorted(glob.glob(path)), desc=f"Processing Files for {year}"):
        df = pd.read_csv(file)
        
        # Convert 'started_at' to datetime
        df['started_at'] = pd.to_datetime(df['started_at'], errors='coerce')
        df = df.dropna(subset=['started_at'])  # Drop rows with invalid dates
        
        # Filter rows for the current year
        df = df[df['started_at'].dt.year == year]
        
        # Append to the yearly DataFrame
        yearly_data = pd.concat([yearly_data, df], ignore_index=True)
    
    # Combine start and end station IDs for the current year
    start_stations = yearly_data['start_station_id'].dropna().unique()
    end_stations = yearly_data['end_station_id'].dropna().unique()
    all_stations = pd.unique(pd.concat([pd.Series(start_stations), pd.Series(end_stations)]))
    
    # Save unique stations for the year into the dictionary
    unique_stations_per_year[year] = all_stations
    
    # Print the number of unique stations for the year
    print(f"Year {year}: {len(all_stations)} unique station IDs")

# Convert the dictionary into a DataFrame
unique_stations_summary = pd.DataFrame({
    'year': list(unique_stations_per_year.keys()),
    'unique_station_count': [len(stations) for stations in unique_stations_per_year.values()]
})

print(f"Summary saved to {output_path}")


Processing year: 2021


Processing Files for 2021:   0%|          | 0/143 [00:00<?, ?it/s]/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_69947/26878111.py:20: DtypeWarning: Columns (5,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)
/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_69947/26878111.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['started_at'] = pd.to_datetime(df['started_at'], errors='coerce')
Processing Files for 2021:   1%|▏         | 2/143 [00:02<02:04,  1.13it/s]/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_69947/26878111.py:20: DtypeWarning: Columns (5,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)
Processing Files for 2021:   2%|▏         | 3/143 [00:03<02:53,  1.24s/it]/var/folders/30/jfztb8191jzff8_xz9

Year 2021: 3241 unique station IDs
Processing year: 2022


Processing Files for 2022:   0%|          | 0/143 [00:00<?, ?it/s]/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_69947/26878111.py:20: DtypeWarning: Columns (5,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)
/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_69947/26878111.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['started_at'] = pd.to_datetime(df['started_at'], errors='coerce')
Processing Files for 2022:   1%|▏         | 2/143 [00:01<01:54,  1.23it/s]/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_69947/26878111.py:20: DtypeWarning: Columns (5,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)
Processing Files for 2022:   2%|▏         | 3/143 [00:03<02:39,  1.14s/it]/var/folders/30/jfztb8191jzff8_xz9

Year 2022: 3574 unique station IDs
Processing year: 2023


Processing Files for 2023:   0%|          | 0/143 [00:00<?, ?it/s]/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_69947/26878111.py:20: DtypeWarning: Columns (5,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)
/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_69947/26878111.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['started_at'] = pd.to_datetime(df['started_at'], errors='coerce')
Processing Files for 2023:   1%|▏         | 2/143 [00:01<01:56,  1.21it/s]/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_69947/26878111.py:20: DtypeWarning: Columns (5,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)
Processing Files for 2023:   2%|▏         | 3/143 [00:03<02:42,  1.16s/it]/var/folders/30/jfztb8191jzff8_xz9

Year 2023: 4378 unique station IDs
Processing year: 2024


Processing Files for 2024:   0%|          | 0/143 [00:00<?, ?it/s]/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_69947/26878111.py:20: DtypeWarning: Columns (5,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)
/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_69947/26878111.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['started_at'] = pd.to_datetime(df['started_at'], errors='coerce')
Processing Files for 2024:   1%|▏         | 2/143 [00:01<01:55,  1.22it/s]/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_69947/26878111.py:20: DtypeWarning: Columns (5,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)
Processing Files for 2024:   2%|▏         | 3/143 [00:03<02:41,  1.15s/it]/var/folders/30/jfztb8191jzff8_xz9

Year 2024: 4587 unique station IDs


NameError: name 'output_path' is not defined

In [106]:
print(unique_stations_summary)

   year  unique_station_count
0  2021                  3241
1  2022                  3574
2  2023                  4378
3  2024                  4587


In [109]:
# Create the unique station count data
data = {
    'year': [2021, 2022, 2023, 2024],
    'unique_station_count': [3241, 3574, 4378, 4587]
}

# Convert to a DataFrame
unique_stations_summary = pd.DataFrame(data)

# Add the number of days in each year (accounting for leap years)
unique_stations_summary['days_in_year'] = unique_stations_summary['year'].apply(
    lambda x: 366 if (x % 4 == 0 and (x % 100 != 0 or x % 400 == 0)) else 365
)

# Calculate the average daily unique stations
unique_stations_summary['avg_daily_unique_stations'] = (
    unique_stations_summary['unique_station_count'] / unique_stations_summary['days_in_year']
)

# Display the result
print(unique_stations_summary)


   year  unique_station_count  days_in_year  avg_daily_unique_stations
0  2021                  3241           365                   8.879452
1  2022                  3574           365                   9.791781
2  2023                  4378           365                  11.994521
3  2024                  4587           366                  12.532787


In [87]:
print(final_activestations_data)

     year  cumulative_week_number    borough  active_stations
0    2021                       1      Bronx              161
1    2021                       1   Brooklyn             1061
2    2021                       1  Manhattan             2098
3    2021                       2      Bronx              166
4    2021                       2   Brooklyn             1118
..    ...                     ...        ...              ...
484  2021                      25     Queens               13
485  2021                      26      Bronx               34
486  2021                      26   Brooklyn              309
487  2021                      26  Manhattan              924
488  2021                      26     Queens                4

[489 rows x 4 columns]


In [80]:
# Group by 'week_number' and 'borough', and aggregate
combined_data_by_borough = final_activestations_data.groupby(['week_number', 'borough'], as_index=False).agg({
    'active_stations': 'sum',  # Sum up active stations for each borough
    'year': 'first'            # Keep the year
})

# Reorder columns to move 'year' to the first position
combined_data_by_borough = combined_data_by_borough[['year', 'week_number', 'borough', 'active_stations']]

# Display the result
print(combined_data_by_borough)


     year  week_number    borough  active_stations
0    2021            1      Bronx              256
1    2021            1   Brooklyn             1617
2    2021            1  Manhattan             3727
3    2021            2      Bronx              227
4    2021            2   Brooklyn             1606
..    ...          ...        ...              ...
787  2024          212     Queens              716
788  2024          213      Bronx              485
789  2024          213   Brooklyn             1863
790  2024          213  Manhattan             3257
791  2024          213     Queens              571

[792 rows x 4 columns]


In [120]:
pipinstall openpyxl

SyntaxError: invalid syntax (1868048066.py, line 1)

In [121]:
import pandas as pd

# Define the file path
file_path = '/Users/amyxqc/Desktop/amy_bikenycfall2024/Data/Active_Stations.csv'  # Replace 'Active_Stations.xlsx' with your actual filename

# Load the data into a DataFrame
df = pd.read_csv(file_path)  # Use read_excel instead of read_csv for Excel files

# Ensure numeric columns are recognized correctly (adjust column names as needed)
numeric_columns = ['2021', '2022', '2023', '2024']  # Replace with your actual numeric column names

 # Compute min, max, mean, std for each year
summary_stats = df[numeric_columns].agg(['min', 'max', 'mean', 'std'])
    
# Display the summary statistics
print(summary_stats)


             2021         2022         2023         2024
min   1217.000000  1545.000000  1321.000000  2078.000000
max   1503.000000  1784.000000  2077.000000  2114.000000
mean  1425.916667  1613.500000  1813.333333  2103.555556
std     96.758799    72.744259   213.511479    11.125546


In [124]:
import pandas as pd

# Define the file path
file_path = '/Users/amyxqc/Desktop/amy_bikenycfall2024/Data/Active_Stations.csv'  # Replace with your actual file path

# Load the data into a DataFrame
df = pd.read_csv(file_path)  # Use read_excel for Excel files

# Ensure numeric columns are recognized correctly (adjust column names as needed)
numeric_columns = ['2021', '2022', '2023', '2024']  # Replace with your actual numeric column names

# Check if the numeric columns exist in the DataFrame
if all(col in df.columns for col in numeric_columns):
    # Flatten the numeric columns into a single series, excluding NaN values
    combined_data = pd.concat([df[col] for col in numeric_columns]).dropna()
    
    # Ensure the combined data is numeric
    combined_data = pd.to_numeric(combined_data, errors='coerce').dropna()
    
    # Compute min, max, mean, std for the combined data
    summary_stats = {
        'min': combined_data.min(),
        'max': combined_data.max(),
        'mean': combined_data.mean(),
        'std': combined_data.std()
    }
    
    # Display the summary statistics
    print("Summary Statistics Across All Years:")
    print(summary_stats)
else:
    print("Some numeric columns are missing. Check column names in the Excel file.")


Summary Statistics Across All Years:
{'min': 1217.0, 'max': 2114.0, 'mean': 1714.7777777777778, 'std': 272.4013490228985}


In [123]:
combined_data = df[numeric_columns].values.flatten()
    
# Compute min, max, mean, std for the combined data
summary_stats = {
    'min': combined_data.min(),
    'max': combined_data.max(),
    'mean': combined_data.mean(),
    'std': combined_data.std()
}
    
# Display the summary statistics
print("Summary Statistics Across All Years:")
print(summary_stats)

Summary Statistics Across All Years:
{'min': nan, 'max': nan, 'mean': nan, 'std': nan}


In [81]:
# Filter the DataFrame to only include rows where 'week_number' is less than or equal to 196
filtered_stationdata = combined_data_by_borough[combined_data_by_borough['week_number'] <= 196]

# Display the filtered DataFrame
print(filtered_stationdata)


     year  week_number    borough  active_stations
0    2021            1      Bronx              256
1    2021            1   Brooklyn             1617
2    2021            1  Manhattan             3727
3    2021            2      Bronx              227
4    2021            2   Brooklyn             1606
..    ...          ...        ...              ...
723  2023          196      Bronx             1379
724  2023          196   Brooklyn             6338
725  2023          196  Manhattan            12582
726  2024          196      Other              110
727  2023          196     Queens             1666

[728 rows x 4 columns]


In [85]:
# Calculate min, max, mean, and std of active stations for each borough
active_station_stats = filtered_stationdata.groupby('borough')['active_stations'].agg(
    min='min',
    max='max',
    mean='mean',
    std='std'
).reset_index()

# Display the result
print(active_station_stats)


         borough  min    max         mean          std
0          Bronx   12   3331   993.830508   644.620303
1       Brooklyn    2  13981  4084.428571  2644.416449
2      Manhattan    2  26860  8275.754098  5069.250288
3          Other    1    110    13.700000    33.862631
4         Queens    2   3594   643.260116   790.598637
5  Staten Island    2     10     6.000000     4.000000


In [83]:
# Define the output file path
output_file_path = '/Users/amyxqc/Desktop/amy_bikenycfall2024/Data/filtered_active_stations.csv'

# Save the filtered data to a CSV file
filtered_stationdata.to_csv(output_file_path, index=False)

# Confirm the file is saved
print(f"Filtered data saved to {output_file_path}")


Filtered data saved to /Users/amyxqc/Desktop/amy_bikenycfall2024/Data/filtered_active_stations.csv


In [22]:
print(df)


                 ride_id  rideable_type started_at   ended_at  \
0       2F0248F2E85771EA  electric_bike 2021-01-19 2021-01-19   
1       49985469DD6C5EC9   classic_bike 2021-01-29 2021-01-29   
2       E3B2362D59B6182D   classic_bike 2021-01-23 2021-01-23   
3       1C82E20D9DB94A58   classic_bike 2021-01-23 2021-01-24   
4       B82510C13F251703   classic_bike 2021-01-09 2021-01-09   
...                  ...            ...        ...        ...   
999995  864CA0FFE468F56D  electric_bike 2021-01-28 2021-01-28   
999996  BB5B74A76763A88C  electric_bike 2021-01-13 2021-01-13   
999997  763DC8B81224A8F9  electric_bike 2021-01-28 2021-01-28   
999998  37E3442B32845217   classic_bike 2021-01-23 2021-01-23   
999999  65A206F6F62DF901  electric_bike 2021-01-14 2021-01-14   

              start_station_name start_station_id  \
0        Rivington St & Ridge St          5406.02   
1            Clark St & Henry St          4789.03   
2            Clark St & Henry St          4789.03   
3      